# Stage 9 — Final Governance, Model Card & Deployment Readiness

**Project:** Heart_Attack_Risk_Assessment  
**Team:** team05 | **Student:** s502  
**Course:** ITI113 | **Semester:** 26S1  
**Environment:** AWS SageMaker Studio + SageMaker-hosted MLflow

## Purpose

Stage 9 is the **final project-governance and closure stage**.

It does not train, tune, recalibrate, or re-evaluate the model.

Instead, it combines evidence from:

- Stage 6 — model selection and threshold freezing;
- Stage 7 — fairness, SHAP and error analysis;
- Stage 8 — one-time locked holdout evaluation.

## Final decision principle

A technically strong model is not automatically approved for unrestricted deployment.

This notebook therefore distinguishes:

- **technical validation**;
- **Responsible-AI findings**;
- **governance status**;
- **deployment readiness**.

The final decision is based on actual saved evidence rather than hard-coded assumptions.

## 1. Install / verify packages

In [1]:
%pip install -q -U mlflow sagemaker-mlflow pandas psutil

print("Packages ready.")

Note: you may need to restart the kernel to use updated packages.
Packages ready.


## 2. Project paths

In [2]:
from pathlib import Path
import json
import os

import boto3
import pandas as pd
import psutil

EXPECTED_PROJECT_FOLDER = "Heart_Attack_Risk_Assessment"
current = Path.cwd().resolve()

if current.name == EXPECTED_PROJECT_FOLDER:
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = next(
        (
            p
            for p in current.parents
            if p.name == EXPECTED_PROJECT_FOLDER
        ),
        None
    )

if PROJECT_ROOT is None:
    fallback = Path(
        "/home/sagemaker-user/Heart_Attack_Risk_Assessment"
    )

    if fallback.exists():
        PROJECT_ROOT = fallback
    else:
        raise FileNotFoundError(
            "Could not locate project root."
        )

CONFIG_DIR = PROJECT_ROOT / "config"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

STAGE7_DIR = ARTIFACT_DIR / "stage7"
STAGE8_DIR = ARTIFACT_DIR / "stage8"
STAGE9_DIR = ARTIFACT_DIR / "stage9"

MODEL_CARD_DIR = STAGE9_DIR / "model_card"
GOVERNANCE_DIR = STAGE9_DIR / "governance"
EVIDENCE_DIR = STAGE9_DIR / "evidence"

for d in [
    MODEL_CARD_DIR,
    GOVERNANCE_DIR,
    EVIDENCE_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True
    )

def memory_mb():
    return (
        psutil.Process(
            os.getpid()
        )
        .memory_info()
        .rss
        / (1024 ** 2)
    )

print("Project root :", PROJECT_ROOT)
print("Stage 9 dir  :", STAGE9_DIR)
print(f"Memory       : {memory_mb():.1f} MB")

Project root : /home/sagemaker-user/Heart_Attack_Risk_Assessment
Stage 9 dir  : /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage9
Memory       : 160.9 MB


## 3. Required evidence files

In [3]:
MLFLOW_CONFIG_FILE = (
    PROJECT_ROOT /
    "mlflow_app_config_team05_s502.json"
)

STAGE6_NOMINATION_FILE = (
    CONFIG_DIR /
    "stage6_candidate_nomination.json"
)

STAGE7_GOVERNANCE_FILE = (
    STAGE7_DIR /
    "governance" /
    "stage7_governance_summary.json"
)

STAGE7_FAIRNESS_FILE = (
    STAGE7_DIR /
    "fairness" /
    "fairness_governance_summary.csv"
)

STAGE7_SHAP_FILE = (
    STAGE7_DIR /
    "shap" /
    "global_shap_importance.csv"
)

STAGE8_SUMMARY_FILE = (
    STAGE8_DIR /
    "governance" /
    "stage8_final_holdout_summary.json"
)

STAGE8_REPORT_FILE = (
    STAGE8_DIR /
    "governance" /
    "stage8_final_report.csv"
)

STAGE8_FAIRNESS_FILE = (
    STAGE8_DIR /
    "fairness" /
    "holdout_fairness_governance_summary.csv"
)

STAGE8_RACE_FILE = (
    STAGE8_DIR /
    "fairness" /
    "holdout_fairness_race.csv"
)

required = {
    "MLflow config":
        MLFLOW_CONFIG_FILE,

    "Stage 6 nomination":
        STAGE6_NOMINATION_FILE,

    "Stage 7 governance":
        STAGE7_GOVERNANCE_FILE,

    "Stage 7 fairness":
        STAGE7_FAIRNESS_FILE,

    "Stage 7 SHAP":
        STAGE7_SHAP_FILE,

    "Stage 8 summary":
        STAGE8_SUMMARY_FILE,

    "Stage 8 final report":
        STAGE8_REPORT_FILE,

    "Stage 8 fairness":
        STAGE8_FAIRNESS_FILE,

    "Stage 8 race detail":
        STAGE8_RACE_FILE,
}

missing = []

for label, path in required.items():

    status = (
        "FOUND"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{status:7} | "
        f"{label:28} | "
        f"{path}"
    )

    if not path.exists():
        missing.append(
            str(path)
        )

if missing:

    raise FileNotFoundError(
        "Stage 9 cannot start because "
        "required evidence is missing:\n"
        +
        "\n".join(
            missing
        )
    )

print(
    "\n[OK] Final governance evidence is ready."
)

FOUND   | MLflow config                | /home/sagemaker-user/Heart_Attack_Risk_Assessment/mlflow_app_config_team05_s502.json
FOUND   | Stage 6 nomination           | /home/sagemaker-user/Heart_Attack_Risk_Assessment/config/stage6_candidate_nomination.json
FOUND   | Stage 7 governance           | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage7/governance/stage7_governance_summary.json
FOUND   | Stage 7 fairness             | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage7/fairness/fairness_governance_summary.csv
FOUND   | Stage 7 SHAP                 | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage7/shap/global_shap_importance.csv
FOUND   | Stage 8 summary              | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage8/governance/stage8_final_holdout_summary.json
FOUND   | Stage 8 final report         | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage8/governance/stage8_final_report.csv
FOUND   

## 4. Load Stage 6, 7 and 8 evidence

In [4]:
with open(
    STAGE6_NOMINATION_FILE,
    "r",
    encoding="utf-8"
) as f:
    stage6 = json.load(f)

with open(
    STAGE7_GOVERNANCE_FILE,
    "r",
    encoding="utf-8"
) as f:
    stage7 = json.load(f)

with open(
    STAGE8_SUMMARY_FILE,
    "r",
    encoding="utf-8"
) as f:
    stage8 = json.load(f)

stage7_fairness = pd.read_csv(
    STAGE7_FAIRNESS_FILE
)

stage8_fairness = pd.read_csv(
    STAGE8_FAIRNESS_FILE
)

stage7_shap = pd.read_csv(
    STAGE7_SHAP_FILE
)

stage8_race = pd.read_csv(
    STAGE8_RACE_FILE
)

stage8_report = pd.read_csv(
    STAGE8_REPORT_FILE
)

print("Stage 6 candidate :", stage6["leading_candidate"])
print("Stage 6 threshold :", stage6["selected_threshold"])
print("Stage 7 status    :", stage7["stage7_status"])
print("Stage 8 status    :", stage8["stage8_status"])

Stage 6 candidate : xgboost
Stage 6 threshold : 0.52
Stage 7 status    : REVIEW REQUIRED
Stage 8 status    : HOLDOUT EVALUATION COMPLETE - FINAL GOVERNANCE REVIEW REQUIRED


## 5. Extract final technical metrics

In [5]:
holdout_metrics = (
    stage8[
        "holdout_metrics"
    ]
)

final_metrics_table = pd.DataFrame([
    {
        "metric":
            "Accuracy",
        "value":
            holdout_metrics[
                "accuracy"
            ],
    },

    {
        "metric":
            "Recall",
        "value":
            holdout_metrics[
                "recall"
            ],
    },

    {
        "metric":
            "Precision",
        "value":
            holdout_metrics[
                "precision"
            ],
    },

    {
        "metric":
            "F1",
        "value":
            holdout_metrics[
                "f1"
            ],
    },

    {
        "metric":
            "PR-AUC",
        "value":
            holdout_metrics[
                "pr_auc"
            ],
    },

    {
        "metric":
            "ROC-AUC",
        "value":
            holdout_metrics[
                "roc_auc"
            ],
    },

    {
        "metric":
            "Brier Score",
        "value":
            holdout_metrics[
                "brier_score"
            ],
    },
])

display(
    final_metrics_table.round(4)
)

,metric,value
0,Accuracy,0.7987
1,Recall,0.7931
2,Precision,0.1921
3,F1,0.3092
4,PR-AUC,0.4198
5,ROC-AUC,0.8847
6,Brier Score,0.1491


## 6. Compare Stage 6 development performance with Stage 8 holdout

In [6]:
comparison = pd.DataFrame([
    {
        "metric":
            "Recall",
        "stage6":
            stage6.get(
                "recall"
            ),
        "stage8":
            holdout_metrics[
                "recall"
            ],
    },

    {
        "metric":
            "Precision",
        "stage6":
            stage6.get(
                "precision"
            ),
        "stage8":
            holdout_metrics[
                "precision"
            ],
    },

    {
        "metric":
            "F1",
        "stage6":
            stage6.get(
                "f1"
            ),
        "stage8":
            holdout_metrics[
                "f1"
            ],
    },

    {
        "metric":
            "PR-AUC",
        "stage6":
            stage6.get(
                "pr_auc"
            ),
        "stage8":
            holdout_metrics[
                "pr_auc"
            ],
    },

    {
        "metric":
            "ROC-AUC",
        "stage6":
            stage6.get(
                "roc_auc"
            ),
        "stage8":
            holdout_metrics[
                "roc_auc"
            ],
    },

    {
        "metric":
            "Brier Score",
        "stage6":
            stage6.get(
                "brier_score"
            ),
        "stage8":
            holdout_metrics[
                "brier_score"
            ],
    },
])

comparison[
    "change"
] = (
    comparison[
        "stage8"
    ]
    -
    comparison[
        "stage6"
    ]
)

display(
    comparison.round(4)
)

comparison.to_csv(
    EVIDENCE_DIR /
    "stage6_vs_stage8_final_comparison.csv",
    index=False
)

,metric,stage6,stage8,change
0,Recall,0.8027,0.7931,-0.0096
1,Precision,0.1958,0.1921,-0.0037
2,F1,0.3148,0.3092,-0.0056
3,PR-AUC,0.4133,0.4198,0.0065
4,ROC-AUC,0.8868,0.8847,-0.0021
5,Brier Score,0.1476,0.1491,0.0015


## 7. Technical validation assessment

This project uses a simple governance interpretation:

- Holdout performance is considered **technically stable** when there is no material collapse relative to Stage 6.
- This is not a regulatory or clinical approval criterion.
- It is an academic project-level acceptance rule.

In [7]:
stage6_recall = float(
    stage6[
        "recall"
    ]
)

stage8_recall = float(
    holdout_metrics[
        "recall"
    ]
)

recall_change = (
    stage8_recall
    -
    stage6_recall
)

stage6_pr_auc = float(
    stage6[
        "pr_auc"
    ]
)

stage8_pr_auc = float(
    holdout_metrics[
        "pr_auc"
    ]
)

pr_auc_change = (
    stage8_pr_auc
    -
    stage6_pr_auc
)

TECHNICAL_STABILITY_RECALL_DROP_LIMIT = 0.05

technical_stable = (
    recall_change
    >=
    -TECHNICAL_STABILITY_RECALL_DROP_LIMIT
)

print(
    "Recall change:",
    round(
        recall_change,
        4
    )
)

print(
    "PR-AUC change:",
    round(
        pr_auc_change,
        4
    )
)

print(
    "Technical stability:",
    (
        "PASS"
        if technical_stable
        else "REVIEW"
    )
)

Recall change: -0.0096
PR-AUC change: 0.0065
Technical stability: PASS


## 8. Final fairness evidence

In [8]:
final_fairness = stage8_fairness[
    [
        "dimension",
        "recall_gap",
        "trigger_threshold",
        "governance_trigger",
    ]
].copy()

display(
    final_fairness.round(4)
)

triggered_dimensions = (
    final_fairness[
        final_fairness[
            "governance_trigger"
        ]
        ==
        "INVESTIGATE"
    ][
        "dimension"
    ]
    .tolist()
)

print(
    "Final fairness triggers:",
    triggered_dimensions
)

,dimension,recall_gap,trigger_threshold,governance_trigger
0,age,0.6391,0.1,INVESTIGATE
1,sex,0.1169,0.1,INVESTIGATE
2,race,0.1668,0.1,INVESTIGATE


Final fairness triggers: ['age', 'sex', 'race']


## 9. Race finding review

Race did not exceed the project recall-gap threshold during Stage 7 OOF analysis, but did exceed it on the locked holdout.

This section records the subgroup detail so the final governance decision does not treat the race finding as a single unexplained number.

In [9]:
display(
    stage8_race[
        [
            "group",
            "n",
            "positive_actual_n",
            "recall",
            "precision",
            "fnr",
            "fpr",
        ]
    ].round(4)
)

,group,n,positive_actual_n,recall,precision,fnr,fpr
0,"Black only, Non-Hispanic",6879,314,0.7611,0.1566,0.2389,0.1960
1,Hispanic,8554,342,0.6725,0.1765,0.3275,0.1307
2,"Multiracial, Non-Hispanic",1912,112,0.8393,0.2260,0.1607,0.1789
3,"Other race only, Non-Hispanic",4453,181,0.8343,0.1953,0.1657,0.1456
4,"White only, Non-Hispanic",63912,3887,0.8032,0.1949,0.1968,0.2148
5,NaN,2704,186,0.7903,0.2082,0.2097,0.2220


## 10. Final SHAP evidence

In [10]:
top_shap = (
    stage7_shap
    .head(15)
    .copy()
)

display(
    top_shap.round(4)
)

top_shap.to_csv(
    EVIDENCE_DIR /
    "top_shap_features.csv",
    index=False
)

,feature,mean_abs_shap
0,cat__HadAngina_No,0.4920
1,"cat__RaceEthnicityCategory_Multiracial, Non-Hi...",0.3923
2,cat__AgeCategory_Age 80 or older,0.3814
3,cat__AgeCategory_Age 18 to 24,0.3538
4,cat__HadDiabetes_Yes,0.3395
5,num__MentalHealthDays,0.3168
6,cat__AgeCategory_Age 40 to 44,0.2085
7,num__PhysicalHealthDays,0.2042
8,cat__AgeCategory_Age 30 to 34,0.2019
9,cat__AgeCategory_Age 45 to 49,0.2009


## 11. Create final risk register

Risk status values:

- **OPEN** — unresolved and must be documented/reviewed;
- **CONTROLLED** — mitigation implemented;
- **ACCEPTED LIMITATION** — known project limitation;
- **CLOSED** — no longer material.

This is an academic governance artifact, not a regulatory certification.

In [11]:
risk_rows = [
    {
        "risk_id":
            "R1",

        "risk":
            "Historical target may be misinterpreted as future risk prediction",

        "evidence":
            "HadHeartAttack is cross-sectional and self-reported",

        "control":
            "Historical-target disclaimer in inference, UI, model card and reports",

        "status":
            "CONTROLLED",
    },

    {
        "risk_id":
            "R2",

        "risk":
            "Age subgroup recall disparity",

        "evidence":
            (
                f"Stage 8 recall gap = "
                f"{float(stage8_fairness.loc[stage8_fairness['dimension']=='age','recall_gap'].iloc[0]):.4f}"
            ),

        "control":
            "Governance review; no unrestricted deployment",

        "status":
            "OPEN",
    },

    {
        "risk_id":
            "R3",

        "risk":
            "Sex subgroup recall disparity",

        "evidence":
            (
                f"Stage 8 recall gap = "
                f"{float(stage8_fairness.loc[stage8_fairness['dimension']=='sex','recall_gap'].iloc[0]):.4f}"
            ),

        "control":
            "Governance review; subgroup monitoring required",

        "status":
            "OPEN",
    },

    {
        "risk_id":
            "R4",

        "risk":
            "Race subgroup recall disparity",

        "evidence":
            (
                f"Stage 8 recall gap = "
                f"{float(stage8_fairness.loc[stage8_fairness['dimension']=='race','recall_gap'].iloc[0]):.4f}"
            ),

        "control":
            "Review subgroup sample sizes and disparity; no unrestricted deployment",

        "status":
            "OPEN",
    },

    {
        "risk_id":
            "R5",

        "risk":
            "High false-positive burden",

        "evidence":
            (
                f"Holdout FP = "
                f"{holdout_metrics['fp']}; "
                f"precision = "
                f"{holdout_metrics['precision']:.4f}"
            ),

        "control":
            "Human review; no automatic clinical action",

        "status":
            "ACCEPTED LIMITATION",
    },

    {
        "risk_id":
            "R6",

        "risk":
            "False negatives remain",

        "evidence":
            (
                f"Holdout FN = "
                f"{holdout_metrics['fn']}; "
                f"recall = "
                f"{holdout_metrics['recall']:.4f}"
            ),

        "control":
            "Educational support only; not a diagnostic replacement",

        "status":
            "ACCEPTED LIMITATION",
    },

    {
        "risk_id":
            "R7",

        "risk":
            "SHAP may be misread as causal evidence",

        "evidence":
            "Stage 7 SHAP analysis completed",

        "control":
            "Explicit non-causal interpretation statement",

        "status":
            "CONTROLLED",
    },

    {
        "risk_id":
            "R8",

        "risk":
            "Geographic and demographic proxies may influence predictions",

        "evidence":
            "State, age, race and sex features appear in SHAP importance",

        "control":
            "Governance documentation and deployment restriction",

        "status":
            "OPEN",
    },

    {
        "risk_id":
            "R9",

        "risk":
            "US BRFSS data may not generalise to Singapore clinical populations",

        "evidence":
            "Dataset represents US survey respondents",

        "control":
            "No Singapore clinical-validity claim",

        "status":
            "ACCEPTED LIMITATION",
    },

    {
        "risk_id":
            "R10",

        "risk":
            "Secrets or sensitive inputs could be exposed operationally",

        "evidence":
            "Academic prototype deployment design",

        "control":
            "Secret management, minimal logs, no raw inference-input logging",

        "status":
            "CONTROLLED",
    },
]

risk_register = pd.DataFrame(
    risk_rows
)

display(
    risk_register
)

risk_register.to_csv(
    GOVERNANCE_DIR /
    "final_risk_register.csv",
    index=False
)

,risk_id,risk,evidence,control,status
0,R1,Historical target may be misinterpreted as fut...,HadHeartAttack is cross-sectional and self-rep...,"Historical-target disclaimer in inference, UI,...",CONTROLLED
1,R2,Age subgroup recall disparity,Stage 8 recall gap = 0.6391,Governance review; no unrestricted deployment,OPEN
2,R3,Sex subgroup recall disparity,Stage 8 recall gap = 0.1169,Governance review; subgroup monitoring required,OPEN
3,R4,Race subgroup recall disparity,Stage 8 recall gap = 0.1668,Review subgroup sample sizes and disparity; no...,OPEN
4,R5,High false-positive burden,Holdout FP = 16756; precision = 0.1921,Human review; no automatic clinical action,ACCEPTED LIMITATION
5,R6,False negatives remain,Holdout FN = 1039; recall = 0.7931,Educational support only; not a diagnostic rep...,ACCEPTED LIMITATION
6,R7,SHAP may be misread as causal evidence,Stage 7 SHAP analysis completed,Explicit non-causal interpretation statement,CONTROLLED
7,R8,Geographic and demographic proxies may influen...,"State, age, race and sex features appear in SH...",Governance documentation and deployment restri...,OPEN
8,R9,US BRFSS data may not generalise to Singapore ...,Dataset represents US survey respondents,No Singapore clinical-validity claim,ACCEPTED LIMITATION
9,R10,Secrets or sensitive inputs could be exposed o...,Academic prototype deployment design,"Secret management, minimal logs, no raw infere...",CONTROLLED


## 12. Determine final governance status

In [12]:
performance_failed = (
    not technical_stable
)

fairness_review_required = (
    len(
        triggered_dimensions
    )
    > 0
)

if performance_failed:

    FINAL_STATUS = (
        "NOT APPROVED - "
        "TECHNICAL PERFORMANCE REVIEW REQUIRED"
    )

elif fairness_review_required:

    FINAL_STATUS = (
        "TECHNICALLY VALIDATED - "
        "GOVERNANCE REVIEW REQUIRED"
    )

else:

    FINAL_STATUS = (
        "TECHNICALLY VALIDATED - "
        "NO PROJECT FAIRNESS TRIGGER"
    )


print(
    "FINAL STATUS:"
)

print(
    FINAL_STATUS
)

FINAL STATUS:
TECHNICALLY VALIDATED - GOVERNANCE REVIEW REQUIRED


## 13. Deployment-readiness decision

Because this is an educational clinical decision-support prototype, the final decision distinguishes between:

- technical readiness for demonstration;
- unrestricted real-world clinical deployment.

The model must not be approved for unsupervised clinical use when material subgroup disparities remain unresolved.

In [13]:
deployment_rows = [
    {
        "area":
            "Model technically stable on holdout",

        "status":
            (
                "YES"
                if technical_stable
                else "NO"
            ),

        "evidence":
            (
                f"Recall change Stage6→Stage8 = "
                f"{recall_change:.4f}"
            ),
    },

    {
        "area":
            "Threshold frozen before holdout",

        "status":
            "YES",

        "evidence":
            stage8[
                "threshold_changed_after_holdout"
            ]
            is False,
    },

    {
        "area":
            "Model retrained on holdout",

        "status":
            "NO",

        "evidence":
            stage8[
                "model_retrained_on_holdout"
            ],
    },

    {
        "area":
            "SHAP completed",

        "status":
            (
                "YES"
                if stage7[
                    "shap_completed"
                ]
                else "NO"
            ),

        "evidence":
            (
                f"Sample size = "
                f"{stage7['shap_sample_size']}"
            ),
    },

    {
        "area":
            "Fairness governance triggers remain",

        "status":
            (
                "YES"
                if fairness_review_required
                else "NO"
            ),

        "evidence":
            ",".join(
                triggered_dimensions
            ),
    },

    {
        "area":
            "Suitable for academic demonstration",

        "status":
            (
                "YES"
                if technical_stable
                else "REVIEW"
            ),

        "evidence":
            "Controlled educational prototype",
    },

    {
        "area":
            "Suitable for unrestricted clinical deployment",

        "status":
            "NO",

        "evidence":
            (
                "Unresolved age/sex/race fairness concerns "
                "and historical self-reported target"
            ),
    },
]

deployment_readiness = pd.DataFrame(
    deployment_rows
)

display(
    deployment_readiness
)

deployment_readiness.to_csv(
    GOVERNANCE_DIR /
    "deployment_readiness.csv",
    index=False
)

,area,status,evidence
0,Model technically stable on holdout,YES,Recall change Stage6→Stage8 = -0.0096
1,Threshold frozen before holdout,YES,True
2,Model retrained on holdout,NO,False
3,SHAP completed,YES,Sample size = 1000
4,Fairness governance triggers remain,YES,"age,sex,race"
5,Suitable for academic demonstration,YES,Controlled educational prototype
6,Suitable for unrestricted clinical deployment,NO,Unresolved age/sex/race fairness concerns and ...


## 14. Create final model card JSON

In [14]:
model_card = {
    "model_name":
        "Heart_Attack_Risk_Assessment_XGBoost",

    "model_type":
        "XGBoost binary classifier",

    "project":
        "Heart_Attack_Risk_Assessment",

    "target":
        "HadHeartAttack",

    "target_interpretation":
        (
            "Historical self-reported heart-attack status "
            "from CDC BRFSS 2022"
        ),

    "prediction_interpretation":
        (
            "Classifies whether a profile resembles respondents "
            "who reported having had a heart attack versus those "
            "who did not."
        ),

    "future_event_prediction":
        False,

    "diagnostic_tool":
        False,

    "intended_use":
        (
            "Educational clinical decision-support prototype "
            "for demonstrating supervised ML, XAI, MLOps and "
            "Responsible-AI evaluation."
        ),

    "prohibited_use":
        [
            "Future heart-attack prediction",
            "Automated diagnosis",
            "Unsupervised treatment decisions",
            "Replacement of clinician judgement",
        ],

    "frozen_threshold":
        float(
            stage6[
                "selected_threshold"
            ]
        ),

    "final_holdout_metrics":
        holdout_metrics,

    "technical_validation":
        (
            "PASS"
            if technical_stable
            else "REVIEW"
        ),

    "stage7_fairness_triggers":
        stage7.get(
            "fairness_triggered_dimensions",
            []
        ),

    "stage8_fairness_triggers":
        triggered_dimensions,

    "shap_completed":
        bool(
            stage7[
                "shap_completed"
            ]
        ),

    "shap_is_causal":
        False,

    "major_limitations":
        [
            "Historical self-reported target",
            "Class imbalance",
            "Low precision / high false-positive burden",
            "Age recall disparity",
            "Sex recall disparity",
            "Race recall disparity on locked holdout",
            "US survey population may not generalise to Singapore",
            "SHAP attribution is not causal explanation",
        ],

    "final_governance_status":
        FINAL_STATUS,

    "deployment_recommendation":
        (
            "Suitable for controlled academic demonstration. "
            "Not approved for unrestricted or unsupervised "
            "clinical deployment."
        ),
}

model_card_json_file = (
    MODEL_CARD_DIR /
    "model_card.json"
)

model_card_json_file.write_text(
    json.dumps(
        model_card,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Saved:",
    model_card_json_file
)

Saved: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage9/model_card/model_card.json


## 15. Create human-readable model card

In [15]:
model_card_md = f'''
# Model Card — Heart Attack Risk Assessment Educational Prototype

## Model
- Model: XGBoost binary classifier
- Target: `HadHeartAttack`
- Frozen threshold: {stage6["selected_threshold"]:.2f}

## Intended interpretation
The model classifies whether an individual's demographic, lifestyle and health characteristics resemble respondents who reported having had a heart attack in the CDC BRFSS 2022 dataset.

It does **not** estimate future heart-attack incidence and does **not** provide a medical diagnosis.

## Final locked-holdout performance
- Accuracy: {holdout_metrics["accuracy"]:.4f}
- Recall: {holdout_metrics["recall"]:.4f}
- Precision: {holdout_metrics["precision"]:.4f}
- F1: {holdout_metrics["f1"]:.4f}
- PR-AUC: {holdout_metrics["pr_auc"]:.4f}
- ROC-AUC: {holdout_metrics["roc_auc"]:.4f}
- Brier Score: {holdout_metrics["brier_score"]:.4f}
- False negatives: {holdout_metrics["fn"]}
- False positives: {holdout_metrics["fp"]}

## Generalisation
Stage 6 recall: {stage6_recall:.4f}  
Stage 8 recall: {stage8_recall:.4f}  
Recall change: {recall_change:.4f}

Stage 6 PR-AUC: {stage6_pr_auc:.4f}  
Stage 8 PR-AUC: {stage8_pr_auc:.4f}  
PR-AUC change: {pr_auc_change:.4f}

## Responsible-AI findings
Stage 7 fairness triggers: {", ".join(stage7.get("fairness_triggered_dimensions", []))}

Stage 8 fairness triggers: {", ".join(triggered_dimensions)}

The locked-holdout subgroup recall gaps were:
{stage8_fairness.to_string(index=False)}

## Explainability
SHAP analysis was completed on a reproducible sample of {stage7["shap_sample_size"]} records.

SHAP values indicate model feature attribution only and do not establish clinical causality.

## Major limitations
- Historical/self-reported target.
- High false-positive burden.
- Remaining false negatives.
- Age subgroup disparity.
- Sex subgroup disparity.
- Race subgroup disparity on final holdout.
- Potential demographic/geographic proxy effects.
- US BRFSS population may not generalise to Singapore.
- Not clinically validated.

## Governance decision
**{FINAL_STATUS}**

## Deployment recommendation
Suitable for controlled academic demonstration and research.

**Not approved for unrestricted or unsupervised clinical deployment.**
'''

model_card_md_file = (
    MODEL_CARD_DIR /
    "model_card.md"
)

model_card_md_file.write_text(
    model_card_md,
    encoding="utf-8"
)

print(
    "Saved:",
    model_card_md_file
)

Saved: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage9/model_card/model_card.md


## 16. Create final governance decision JSON

In [16]:
final_governance_decision = {
    "project":
        "Heart_Attack_Risk_Assessment",

    "selected_model":
        stage6[
            "leading_candidate"
        ],

    "frozen_threshold":
        stage6[
            "selected_threshold"
        ],

    "technical_validation":
        (
            "PASS"
            if technical_stable
            else "REVIEW"
        ),

    "final_holdout_recall":
        holdout_metrics[
            "recall"
        ],

    "final_holdout_pr_auc":
        holdout_metrics[
            "pr_auc"
        ],

    "final_holdout_roc_auc":
        holdout_metrics[
            "roc_auc"
        ],

    "final_holdout_brier_score":
        holdout_metrics[
            "brier_score"
        ],

    "fairness_triggered_dimensions":
        triggered_dimensions,

    "unresolved_governance_risks":
        risk_register[
            risk_register[
                "status"
            ]
            ==
            "OPEN"
        ][
            "risk"
        ]
        .tolist(),

    "final_status":
        FINAL_STATUS,

    "academic_demo_ready":
        bool(
            technical_stable
        ),

    "unrestricted_clinical_deployment_approved":
        False,

    "reason":
        (
            "Technical performance remained stable on the "
            "one-time locked holdout, but material subgroup "
            "recall disparities remain for age, sex and race."
        ),

    "historical_target_disclaimer":
        (
            "HadHeartAttack is a historical self-reported target. "
            "The system does not predict future heart attacks."
        ),

    "project_closed":
        True,
}

final_governance_file = (
    GOVERNANCE_DIR /
    "final_governance_decision.json"
)

final_governance_file.write_text(
    json.dumps(
        final_governance_decision,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        final_governance_decision,
        indent=2
    )
)

{
  "project": "Heart_Attack_Risk_Assessment",
  "selected_model": "xgboost",
  "frozen_threshold": 0.52,
  "technical_validation": "PASS",
  "final_holdout_recall": 0.7931103146156909,
  "final_holdout_pr_auc": 0.4198124691659367,
  "final_holdout_roc_auc": 0.8846650440765925,
  "final_holdout_brier_score": 0.1491195112466812,
  "fairness_triggered_dimensions": [
    "age",
    "sex",
    "race"
  ],
  "unresolved_governance_risks": [
    "Age subgroup recall disparity",
    "Sex subgroup recall disparity",
    "Race subgroup recall disparity",
    "Geographic and demographic proxies may influence predictions"
  ],
  "final_status": "TECHNICALLY VALIDATED - GOVERNANCE REVIEW REQUIRED",
  "academic_demo_ready": true,
  "unrestricted_clinical_deployment_approved": false,
  "reason": "Technical performance remained stable on the one-time locked holdout, but material subgroup recall disparities remain for age, sex and race.",
  "historical_target_disclaimer": "HadHeartAttack is a historic

## 17. Create final project evidence table

In [17]:
final_evidence = pd.DataFrame([
    {
        "stage":
            "Stage 6",
        "evidence":
            "Candidate model selected",
        "result":
            stage6[
                "leading_candidate"
            ],
    },

    {
        "stage":
            "Stage 6",
        "evidence":
            "Frozen threshold",
        "result":
            stage6[
                "selected_threshold"
            ],
    },

    {
        "stage":
            "Stage 7",
        "evidence":
            "SHAP completed",
        "result":
            stage7[
                "shap_completed"
            ],
    },

    {
        "stage":
            "Stage 7",
        "evidence":
            "Fairness triggers",
        "result":
            ",".join(
                stage7.get(
                    "fairness_triggered_dimensions",
                    []
                )
            ),
    },

    {
        "stage":
            "Stage 8",
        "evidence":
            "Holdout recall",
        "result":
            holdout_metrics[
                "recall"
            ],
    },

    {
        "stage":
            "Stage 8",
        "evidence":
            "Holdout PR-AUC",
        "result":
            holdout_metrics[
                "pr_auc"
            ],
    },

    {
        "stage":
            "Stage 8",
        "evidence":
            "Holdout ROC-AUC",
        "result":
            holdout_metrics[
                "roc_auc"
            ],
    },

    {
        "stage":
            "Stage 8",
        "evidence":
            "Holdout fairness triggers",
        "result":
            ",".join(
                triggered_dimensions
            ),
    },

    {
        "stage":
            "Stage 9",
        "evidence":
            "Final governance status",
        "result":
            FINAL_STATUS,
    },

    {
        "stage":
            "Stage 9",
        "evidence":
            "Unrestricted clinical deployment",
        "result":
            "NOT APPROVED",
    },
])

display(
    final_evidence
)

final_evidence.to_csv(
    EVIDENCE_DIR /
    "final_project_evidence.csv",
    index=False
)

,stage,evidence,result
0,Stage 6,Candidate model selected,xgboost
1,Stage 6,Frozen threshold,0.52
2,Stage 7,SHAP completed,True
3,Stage 7,Fairness triggers,"age,sex"
4,Stage 8,Holdout recall,0.79311
5,Stage 8,Holdout PR-AUC,0.419812
6,Stage 8,Holdout ROC-AUC,0.884665
7,Stage 8,Holdout fairness triggers,"age,sex,race"
8,Stage 9,Final governance status,TECHNICALLY VALIDATED - GOVERNANCE REVIEW REQU...
9,Stage 9,Unrestricted clinical deployment,NOT APPROVED


## 18. Save compact final model decision

In [18]:
final_model_decision = {
    "model":
        stage6[
            "leading_candidate"
        ],

    "threshold":
        stage6[
            "selected_threshold"
        ],

    "holdout_recall":
        holdout_metrics[
            "recall"
        ],

    "holdout_pr_auc":
        holdout_metrics[
            "pr_auc"
        ],

    "holdout_roc_auc":
        holdout_metrics[
            "roc_auc"
        ],

    "governance_status":
        FINAL_STATUS,

    "academic_demo_ready":
        bool(
            technical_stable
        ),

    "clinical_deployment_approved":
        False,
}

final_model_decision_file = (
    CONFIG_DIR /
    "final_model_decision.json"
)

final_model_decision_file.write_text(
    json.dumps(
        final_model_decision,
        indent=2
    ),
    encoding="utf-8"
)

print(
    "Saved:",
    final_model_decision_file
)

Saved: /home/sagemaker-user/Heart_Attack_Risk_Assessment/config/final_model_decision.json


## 19. Connect to Team05 MLflow

In [19]:
import mlflow

with open(
    MLFLOW_CONFIG_FILE,
    "r",
    encoding="utf-8"
) as f:
    mlflow_config = json.load(f)

REGION = mlflow_config["REGION"]
MLFLOW_APP_ARN = mlflow_config["MLFLOW_APP_ARN"]
MLFLOW_EXPERIMENT = mlflow_config["EXPERIMENT_NAME"]
TEAM_ID = mlflow_config["TEAM_ID"]
STUDENT_ID = mlflow_config["STUDENT_ID"]
PROJECT_NAME = mlflow_config["PROJECT_NAME"]

mlflow.set_tracking_uri(
    MLFLOW_APP_ARN
)

mlflow.set_experiment(
    MLFLOW_EXPERIMENT
)

print(
    "MLflow experiment:",
    MLFLOW_EXPERIMENT
)

MLflow experiment: ITI113/team05/Experiment1


## 20. Log final Stage 9 governance evidence to MLflow

In [20]:
with mlflow.start_run(
    run_name=(
        f"{TEAM_ID}_"
        f"{STUDENT_ID}_"
        f"stage9_final_governance"
    )
) as run:

    mlflow.set_tags({
        "team_id":
            TEAM_ID,

        "student_id":
            STUDENT_ID,

        "project_name":
            PROJECT_NAME,

        "stage":
            "stage9_final_governance",

        "model":
            stage6[
                "leading_candidate"
            ],

        "final_status":
            FINAL_STATUS,
    })

    mlflow.log_params({
        "frozen_threshold":
            stage6[
                "selected_threshold"
            ],

        "technical_stable":
            technical_stable,

        "fairness_trigger_count":
            len(
                triggered_dimensions
            ),

        "academic_demo_ready":
            technical_stable,

        "clinical_deployment_approved":
            False,

        "project_closed":
            True,
    })

    mlflow.log_metrics({
        "final_holdout_recall":
            holdout_metrics[
                "recall"
            ],

        "final_holdout_precision":
            holdout_metrics[
                "precision"
            ],

        "final_holdout_f1":
            holdout_metrics[
                "f1"
            ],

        "final_holdout_pr_auc":
            holdout_metrics[
                "pr_auc"
            ],

        "final_holdout_roc_auc":
            holdout_metrics[
                "roc_auc"
            ],

        "final_holdout_brier":
            holdout_metrics[
                "brier_score"
            ],

        "final_fairness_trigger_count":
            float(
                len(
                    triggered_dimensions
                )
            ),
    })

    for folder in [
        MODEL_CARD_DIR,
        GOVERNANCE_DIR,
        EVIDENCE_DIR,
    ]:

        mlflow.log_artifacts(
            str(folder),
            artifact_path=(
                f"stage9/"
                f"{folder.name}"
            )
        )

    mlflow.log_artifact(
        str(
            final_model_decision_file
        ),
        artifact_path="stage9/config"
    )

    print(
        "Stage 9 MLflow Run ID:",
        run.info.run_id
    )

Stage 9 MLflow Run ID: 52280b3f02564a69bae09d5d36f177ae
🏃 View run team05_s502_stage9_final_governance at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/52280b3f02564a69bae09d5d36f177ae
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1


# Final Stage 9 Output

After this notebook completes, the final project package will contain:

```text
artifacts/stage9/
├── model_card/
│   ├── model_card.json
│   └── model_card.md
│
├── governance/
│   ├── final_governance_decision.json
│   ├── final_risk_register.csv
│   └── deployment_readiness.csv
│
└── evidence/
    ├── final_project_evidence.csv
    ├── stage6_vs_stage8_final_comparison.csv
    └── top_shap_features.csv

config/
└── final_model_decision.json
```

## Final interpretation

The project can conclude:

- XGBoost was technically validated on the locked holdout;
- aggregate performance remained stable;
- Responsible-AI analysis identified material subgroup recall disparities;
- the prototype is suitable for controlled academic demonstration;
- unrestricted or unsupervised clinical deployment is not approved;
- the target remains historical/self-reported and is not a future-event prediction target.

No further model tuning or holdout evaluation should occur after Stage 9.